In [1]:
import gradio as gr
import pandas as pd
import pickle

c:\Users\Aya\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from transformers import pipeline

sentiment_pipeline = pipeline(
    "sentiment-analysis",
    model="cardiffnlp/twitter-roberta-base-sentiment"
)
rules_fp = pickle.load(open("rules_fp.pkl", "rb"))

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 17822.82it/s]


In [3]:
def bert_sentiment(text):
    """
    Predict sentiment using BERT model
    Returns clean label + confidence in a table format
    """

    result = sentiment_pipeline(str(text)[:512])[0]

    label = result["label"]
    score = round(result["score"], 3)

    if label == "LABEL_0":
        sentiment = "negative"
    elif label == "LABEL_1":
        sentiment = "neutral"
    else:
        sentiment = "positive"

    return pd.DataFrame(
    [[text, sentiment, score]],
    columns=["Text", "Sentiment", "Confidence"]
)

In [4]:
def association_recommend(items):

    if rules_fp is None or len(rules_fp) == 0:
        return pd.DataFrame([["No rules loaded"]], columns=["Recommended Products"])

    items = str(items).lower()
    recommendations = []

    for _, row in rules_fp.iterrows():
        antecedents = str(row['antecedents']).lower()

        if any(item.strip() in antecedents for item in items.split(",")):
            recommendations.extend(list(row['consequents']))

        if len(recommendations) >= 5:
            break

    if not recommendations:
        recommendations = ["No recommendations found"]

    return pd.DataFrame(recommendations, columns=["Recommended Products"])

In [5]:
def system(input_text, model_type):

    if model_type == "Sentiment Analysis (BERT)":
        return bert_sentiment(input_text)
    else:
        return association_recommend(input_text)

In [6]:
custom_css = """ 
body, .gradio-container {
    background: linear-gradient(135deg, #2b0610, #6a0f1f) !important;
}

#main-box {
    background: #0f0f0f;
    border-radius: 20px;
    padding: 30px;
    box-shadow: 0 0 30px rgba(150, 20, 50, 0.6);
}

h1 {
    color: #ffffff !important;
    text-align: center;
    font-size: 40px !important;
    font-weight: 800;
}

h2, h3, label {
    color: #ffffff !important;
}

textarea, input {
    background: #000000 !important;
    color: white !important;
    border: 1px solid #333 !important;
    border-radius: 12px !important;
}

textarea:focus, input:focus {
    border: 1px solid #ff4d6d !important;
    outline: none !important;
}

.my-table {
    border: 1px solid #333 !important;
    border-radius: 12px !important;
    overflow: hidden !important;
    background: #0f0f0f !important;
}

.my-table .styler {
    background: transparent !important;
}

.my-table table {
    border-collapse: collapse !important;
}

.my-table td, .my-table th {
    border: 1px solid #222 !important;
    color: white !important;
}

.my-table button.icon {
    background-color: #e0e0e0 !important;
    color: #000 !important;
    border-radius: 50% !important;
    opacity: 1 !important;
    box-shadow: none !important;
    transform: none !important;
    transition: 0.3s ease;
}

.my-table button.icon:hover {
    background-color: #8b1c2c !important;
    color: white !important;
}

#main-run-btn {
    background: #6a0f1f !important;
    color: white !important;
    border-radius: 14px !important;
    font-size: 18px !important;
    font-weight: bold !important;
    padding: 12px !important;
    border: 1px solid #8b1c2c !important;
    cursor: pointer !important;
    width: 100%;
    transition: all 0.4s cubic-bezier(0.4, 0, 0.2, 1) !important;
}

#main-run-btn:hover {
    background: #a3263a !important;
    border-color: #ff4d6d !important;
    box-shadow: 0 0 20px rgba(163, 38, 58, 0.6), 0 0 40px rgba(139, 28, 44, 0.4) !important;
    transform: translateY(-3px) scale(1.01) !important;
}

#main-run-btn:active {
    transform: translateY(0) scale(0.98) !important;
}

input[type="radio"] {
    appearance: none;
    -webkit-appearance: none;
    width: 18px;
    height: 18px;
    border: 2px solid #444;
    border-radius: 50%;
    background-color: #000000;
    cursor: pointer;
    position: relative;
    transition: 0.2s ease-in-out;
}

input[type="radio"]:checked {
    border-color: #8b1c2c;
    box-shadow: 0 0 10px rgba(139, 28, 44, 0.8);
}

input[type="radio"]:checked::before {
    content: "";
    width: 8px;
    height: 8px;
    background-color: #8b1c2c;
    border-radius: 50%;
    position: absolute;
    top: 50%;
    left: 50%;
    transform: translate(-50%, -50%);
}
""" 

In [7]:
with gr.Blocks(css=custom_css, fill_width=True) as gui:

    gr.Markdown("""
<div style="text-align:center;">

<h1 style="
    font-size:48px;
    color:white;
    margin-bottom:10px;
    font-weight:800;
">
🛒 E-Commerce AI System
</h1>

<p style="
    font-size:22px;
    color:white;
    margin-top:0px;
    opacity:0.9;
">
Sentiment Analysis + Recommendation System
</p>

</div>
""")

    with gr.Column(elem_id="main-box"):

        with gr.Row():

            with gr.Column(scale=1):

                input_box = gr.Textbox(
                    label="Enter Text / Products",
                    placeholder="e.g. I love this product OR milk, bread",
                    lines=4
                )

                model_choice = gr.Radio(
                    ["Sentiment Analysis (BERT)", "Association Rules"],
                    label="Choose AI Model"
                )

                
                run_btn = gr.Button(" Get Results 📊", elem_id="main-run-btn")

            with gr.Column(scale=2):

                output_table = gr.Dataframe(
                label="AI Output",
                interactive=False,
                elem_classes="my-table"
                 )

    run_btn.click(
        fn=system,
        inputs=[input_box, model_choice],
        outputs=output_table
    )

C:\Users\Aya\AppData\Local\Temp\ipykernel_17628\2085317865.py:1: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: css. Please pass these parameters to launch() instead.
  with gr.Blocks(css=custom_css, fill_width=True) as gui:


In [8]:
gui.launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
